In [4]:
import pandas as pd
from typing import Optional

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [16]:
normal_ranges = {'column_1': {"between": [1, 3]},
                 'column_2': {"ge": 2, "replacement": pd.NA},
                 'column_3': {"le": 2, "replacement": -9999}
                }


df = pd.DataFrame({"column_1": [-1, 0, 1, 2, 3, 4, 5, pd.NA], #between
                   "column_2": [-1, 0, 1, 2, 3, 4, 5, pd.NA], # ge
                   "column_3": [-1, 0, 1, 2, 3, 4, 5, pd.NA], # le
                  })

def replace_anomalous(df: pd.DataFrame, 
                      normal_ranges: dict[str, dict[str, Optional[any]]]) -> pd.DataFrame:
    """
    Replace anomalous values in a DataFrame based on the specified configuration.
    By default, anomalous values are replaced with pd.NA unless specified otherwise 
    by the 'value' key.
    NB: The limits for 'between' are inclusive of start and end

    Parameters:
    - df (pd.DataFrame): The input DataFrame.
    - normal_ranges (Dict[str, Dict[str, Optional[any]]]): A dictionary specifying valid ranges 
        for each column and what to replace anomalous values with.
        If no replacement value is provided, pd.NA will be used by default.

    Returns:
    pd.DataFrame: A new DataFrame with anomalous values replaced according to the configuration.

    Raises:
    ValueError: If an invalid range type or an invalid 'between' range is provided in the configuration.

    Example:
    ```python
    import pandas as pd

    data = {'A': [1, 5, 10, 8], 
            'B': [5, 15, 25, 30]}
    df = pd.DataFrame(data)

    normal_ranges = {'A': {'between': (3, 9)}, 
              'B': {'ge': 20, 'replacement': 888}}

    result_df = replace_anomalous(df, normal_ranges)
    ```
    """
    
    df_out = df.copy()
    for column, config in normal_ranges.items():

        # Determine which type was selected.
        range_type = next((key for key in ["between", "ge", "le"] 
                           if key in config), None)
        if range_type is None:
            raise ValueError("Invalid range type")
            
        limit = config[range_type]
        replacement = config.get("replacement", pd.NA)

        # Replace values outside the specified normative range with the replacement value
        match range_type:
            case "between":
                valid_range = config[range_type]
                if len(valid_range) != 2:
                    raise ValueError("Invalid 'between' range")

                is_anomalous = ~df_out[column].between(*config[range_type])
            case "ge":
                is_anomalous = ~df_out[column].ge(config[range_type])
            case "le":
                is_anomalous = ~df_out[column].le(config[range_type])
            case _:
                raise ValueError()

        df_out.loc[is_anomalous, column] = replacement

    return df_out

In [17]:
df.column_1.between(2,3)

0    False
1    False
2    False
3     True
4     True
5    False
6    False
7    False
Name: column_1, dtype: bool

In [18]:
df

normal_ranges

replace_anomalous(df, normal_ranges)

,column_1,column_2,column_3
0,-1,-1,-1
1,0,0,0
2,1,1,1
3,2,2,2
4,3,3,3
5,4,4,4
6,5,5,5
7,<NA>,<NA>,<NA>


{'column_1': {'between': [1, 3]},
 'column_2': {'ge': 2, 'replacement': <NA>},
 'column_3': {'le': 2, 'replacement': -9999}}

,column_1,column_2,column_3
0,<NA>,<NA>,-1
1,<NA>,<NA>,0
2,1,<NA>,1
3,2,2,2
4,3,3,-9999
5,<NA>,4,-9999
6,<NA>,5,-9999
7,<NA>,<NA>,-9999
